# Experiment 4 — Tree model (gradient boosting) on the 71 features

Logistic regression treats every feature on its own. A tree model can learn **combinations**
(e.g. "lower TTR matters only when surprisal also drops").

We use scikit-learn's `HistGradientBoostingClassifier` (no extra install), on the **71 features only**
(no TF-IDF — trees are slow on 20,000 sparse text columns), and compare with logistic regression on
the same 71 features from Experiment 3.

| File | Features | Scored on |
|---|---|---|
| `cumulative`, `step` | 71 `delta_*` (B − A) | held-out pairs |
| `absolute` | 71 raw features | held-out rows **and** held-out pairs (`score(edit row) > score(base row)`) |

Same 20% test articles. Results go to `output/paragraph/tables/exp4-tree-model.csv`.

## 1. Settings

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)

LEVEL = "paragraph"  # "paragraph" or "sentence" - which data/<LEVEL>/master/ to read, output/<LEVEL>/tables/ to write
master_dir = Path(f"../../data/{LEVEL}/master")  # master sheets + feature-engineering/
tables_dir = Path(f"../../output/{LEVEL}/tables")  # final result tables
results_path = tables_dir / "exp4-tree-model.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)

SEED = 42
TEST_SIZE = 0.2

# These 4 columns identify which base/edit pair a row of master-absolute.csv belongs to
PAIR_KEY = ["article", "revision_group_no", "pair_older_depth", "pair_newer_depth"]

ALL_FEATURES = pd.read_csv(master_dir / "feature-engineering" / "final_features.csv")["feature"].tolist()
print(len(ALL_FEATURES), "features")

71 features


## 2. Train / test split (by article)

In [2]:
# one shared list of test articles for both levels, so paragraph and sentence results are measured on the same
# articles. Chosen once from the paragraph master sheet with the same SEED / TEST_SIZE as before (so the paragraph
# split is unchanged) and saved; every experiment at every level reads it from then on.
test_articles_file = Path("../../data/test-articles.csv")
if not test_articles_file.exists():
    paragraph_articles = pd.read_csv("../../data/paragraph/master/master-absolute.csv", usecols=["article"])["article"].astype(str)
    unique_paragraph_articles = np.array(sorted(paragraph_articles.unique()))
    n_test = int(round(len(unique_paragraph_articles) * TEST_SIZE))
    chosen = sorted(np.random.default_rng(SEED).permutation(unique_paragraph_articles)[:n_test])
    pd.DataFrame({"article": chosen}).to_csv(test_articles_file, index=False)
test_articles = set(pd.read_csv(test_articles_file)["article"].astype(str))
articles = pd.read_csv(master_dir / "master-absolute.csv", usecols=["article"])["article"].astype(str)
unique_articles = np.array(sorted(articles.unique()))
print("test articles:", len(test_articles & set(unique_articles)), "of", len(unique_articles))

# paragraphs where a metric could not be calculated (one word headings like "Steps") - made by feature-engineering.ipynb
# every pair that has one of these paragraphs (as base or as edit) is left out
undefined = pd.read_csv(master_dir / "feature-engineering" / "undefined_paragraphs.csv")
undefined_keys = set(zip(undefined["article"].astype(str), undefined["revision_group_no"], undefined["revision_depth"]))
print("undefined paragraphs:", len(undefined_keys))

def defined_pairs(df, base_depth_column, edit_depth_column):
    #True for every pair where both paragraphs have all metrics
    base_keys = zip(df["article"].astype(str), df["revision_group_no"], df[base_depth_column])
    edit_keys = zip(df["article"].astype(str), df["revision_group_no"], df[edit_depth_column])
    return np.array([base not in undefined_keys and edit not in undefined_keys for base, edit in zip(base_keys, edit_keys)])

test articles: 1504 of 7519
undefined paragraphs: 500


## 3. Helpers

Trees don't need scaling, so we pass the numbers in as they are.

In [3]:
def new_model():
    return HistGradientBoostingClassifier(
        max_iter=500,          # up to 500 trees
        learning_rate=0.1,
        early_stopping=True,   # stops when a 10% validation slice stops improving
        random_state=SEED,
    )


def train_and_test(X_train, y_train, X_test, y_test):
    model = new_model()
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, predicted).ravel()
    metrics = {
        "accuracy": accuracy_score(y_test, predicted),
        "precision": precision_score(y_test, predicted),
        "recall": recall_score(y_test, predicted),
        "f1": f1_score(y_test, predicted),
        "auc": roc_auc_score(y_test, probability),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "n_trees": model.n_iter_,
    }
    return metrics, model


def show(name, m):
    print(f"{name:30s} acc={m['accuracy']:.4f}  prec={m['precision']:.4f}  rec={m['recall']:.4f}  "
          f"f1={m['f1']:.4f}  auc={m['auc']:.4f}  trees={m['n_trees']}")


results = []

## 4. Pair files (cumulative and step)

In [4]:
for setting in ["cumulative", "step"]:
    delta_cols = ["delta_" + f for f in ALL_FEATURES]
    df = pd.read_csv(master_dir / f"master-pairs-{setting}.csv",
                     usecols=["article", "revision_group_no", "base_depth", "edit_depth", *delta_cols], low_memory=False)
    keep = defined_pairs(df, "base_depth", "edit_depth")
    print("left out (pair has an undefined paragraph):", (~keep).sum(), "pairs")
    df = df[keep].reset_index(drop=True)

    # same random swap as the other experiments; label = B is the edit
    swap = np.random.default_rng(SEED).integers(0, 2, size=len(df)).astype(bool)
    y = (~swap).astype(int)
    sign = np.where(swap, -1.0, 1.0)
    X = df[delta_cols].fillna(0).to_numpy(dtype=float) * sign[:, None]

    is_test = df["article"].astype(str).isin(test_articles).to_numpy()
    is_train = ~is_test

    m, model = train_and_test(X[is_train], y[is_train], X[is_test], y[is_test])
    show(setting, m)
    results.append({"file": setting, "model": "tree", **m})

left out (pair has an undefined paragraph): 541 pairs


cumulative                     acc=0.7081  prec=0.7120  rec=0.6999  f1=0.7059  auc=0.7824  trees=257


left out (pair has an undefined paragraph): 496 pairs


step                           acc=0.6921  prec=0.6940  rec=0.6919  f1=0.6929  auc=0.7658  trees=169


## 5. Absolute file

Each row is one paragraph, label `edit` = 1, `base` = 0. Scored two ways:

1. `absolute_rows` — each test row on its own (capped low: middle versions appear twice with opposite labels).
2. `absolute_pairs` — every base/edit pair is two rows here (same `PAIR_KEY`); a pair is correct when
   `score(edit row) > score(base row)`.

In [5]:
df = pd.read_csv(master_dir / "master-absolute.csv",
                 usecols=["label", *PAIR_KEY, *ALL_FEATURES], low_memory=False)
keep = defined_pairs(df, "pair_older_depth", "pair_newer_depth")
print("left out (pair has an undefined paragraph):", (~keep).sum(), "rows")
df = df[keep].reset_index(drop=True)
X = df[ALL_FEATURES].fillna(0).to_numpy(dtype=float)
y = (df["label"] == "edit").astype(int).to_numpy()
is_test = df["article"].astype(str).isin(test_articles).to_numpy()
is_train = ~is_test

m, model = train_and_test(X[is_train], y[is_train], X[is_test], y[is_test])
show("absolute_rows", m)
results.append({"file": "absolute_rows", "model": "tree", **m})

# pair accuracy: put base and edit score of the same pair side by side, then compare
test_rows = df.loc[is_test, PAIR_KEY + ["label"]].copy()
test_rows["score"] = model.predict_proba(X[is_test])[:, 1]
table = test_rows.pivot_table(index=PAIR_KEY, columns="label", values="score")
pair_acc = (table["edit"] > table["base"]).mean()
print(f"absolute_pairs  acc={pair_acc:.4f}  (n={len(table)})")
results.append({"file": "absolute_pairs", "model": "tree", "accuracy": pair_acc})

left out (pair has an undefined paragraph): 992 rows


absolute_rows                  acc=0.5557  prec=0.5444  rec=0.6831  f1=0.6059  auc=0.5907  trees=47
absolute_pairs  acc=0.5647  (n=38686)


## 6. Results: tree vs logistic regression (same 71 features)

In [ ]:
results_df = pd.DataFrame(results)
results_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_path, index=False)
print("saved", results_path)
results_df.round(4)

In [7]:
# Logistic regression numbers from Experiment 3 (features_only and tfidf+all_features)
exp3 = pd.read_csv(tables_dir / "exp3-tfidf-all-features.csv")
exp3 = exp3[exp3["feature_set"].isin(["features_only", "tfidf+all_features"])]

columns = ["absolute_rows", "absolute_pairs", "cumulative", "step"]
comparison = pd.DataFrame({
    "logistic (71 features)": exp3[exp3["feature_set"] == "features_only"].set_index("file")["accuracy"],
    "tree (71 features)": results_df.set_index("file")["accuracy"],
    "logistic (TF-IDF + 71)": exp3[exp3["feature_set"] == "tfidf+all_features"].set_index("file")["accuracy"],
}).loc[columns].T
comparison.round(4)

file,absolute_rows,absolute_pairs,cumulative,step
logistic (71 features),NaN,NaN,0.6814,0.6591
tree (71 features),0.5557,0.5647,0.7081,0.6921
logistic (TF-IDF + 71),NaN,NaN,0.6960,0.6750
